In [1]:
import json
import pandas as pd
from collections import defaultdict
import os

In [2]:
data_folder = "data/WestAsia2026/analysis/Lebanon/"

In [3]:
with open(data_folder + "answers.json", "r") as f:
    answers = json.load(f)

answers_df = pd.DataFrame(answers)

In [4]:
with open(data_folder + "key_indicator_numbers.json", "r") as f:
    key_indicator_numbers = json.load(f)

key_indicator_numbers_df = pd.DataFrame(key_indicator_numbers)

In [5]:
with open(data_folder + "risk_list.json", "r") as f:
    risk_list = json.load(f)

risks_df = pd.DataFrame(risk_list)

In [6]:
biggest_risks_numbers = key_indicator_numbers_df[key_indicator_numbers_df["risk_score"] >= 9]
biggest_risks_numbers

,task,pillar,subpillar,sector,key_indicator,number,unit,location,specific_population,date,risk_score,ID
20,situation_analysis_1d,Humanitarian Access,Relief To Population,-,Attacks on emergency medical service workers,67,incidents,Lebanon,medical staff,-,9,"[85, 1046]"
21,situation_analysis_1d,Humanitarian Access,Relief To Population,-,Paramedics killed in attack,1,people,"Majdal Zoun, Lebanon",paramedics,09-03-2026,9,"[232, 1716]"
22,situation_analysis_2d,Humanitarian Conditions,Living Standards,Education,Children unable to attend classes in public sc...,400000,children,Lebanon,public school children,-,9,"[1045, 84]"
23,situation_analysis_2d,Humanitarian Conditions,Living Standards,Education,Schools used as collective shelters,449,schools,Lebanon,-,-,9,[2972]
36,situation_analysis_2d,Humanitarian Conditions,Living Standards,Livelihoods,Unaffordable rents,250,USD,North,Syrian refugees,-,9,[1779]
49,situation_analysis_2d,Humanitarian Conditions,Living Standards,Protection,Overcrowded shelters,-,-,Lebanon,-,-,9,"[2821, 573]"
50,situation_analysis_2d,Humanitarian Conditions,Living Standards,Protection,Lack of gender-segregated WASH facilities,-,-,Lebanon,"women, girls",-,9,"[1777, 1811]"
78,situation_analysis_2d,Humanitarian Conditions,Coping Mechanisms,WASH,Exposure to waterborne diseases,-1,people at risk,Lebanon,children,-,9,[129]
92,situation_analysis_2d,Humanitarian Conditions,Physical And Mental Well Being,Logistics,Civilian fatalities in Arkay airstrike,9,people,Arkay,including 5 children,22-03-2023,9,[2873]
93,situation_analysis_2d,Humanitarian Conditions,Physical And Mental Well Being,Logistics,Fatalities in Nmairye airstrike,9,people,Nmairye,-,-,9,[2873]


In [7]:
key_indicator_numbers_df[key_indicator_numbers_df.pillar=="Displacement"]

,task,pillar,subpillar,sector,key_indicator,number,unit,location,specific_population,date,risk_score,ID
0,situation_analysis_1d,Displacement,Push Factors,-,Territory under evacuation orders,14,%,Lebanon,-,-,8,[2436]
1,situation_analysis_1d,Displacement,Push Factors,-,Territory under evacuation orders,20,%,Lebanon,-,-,8,[1496]
2,situation_analysis_1d,Displacement,Intentions,-,Persons crossing from Lebanon into Syria,147800,people,Lebanon-Syria border,displaced Syrians and others,-,6,[1659]
3,situation_analysis_1d,Displacement,Intentions,-,Syrians indicating intention to return permane...,39840,people,Lebanon-Syria border,Syrian refugees,-,6,[1659]
4,situation_analysis_1d,Displacement,Local Integration,-,Rising community tensions,-,-,Lebanon,displaced populations and host communities,-,7,"[2181, 2399]"


In [8]:
sectors_risks = risks_df[risks_df.task=="situation_analysis_2d"]
sectors_list = sectors_risks.sector.unique()
sectors_list

array(['Education', 'Food Security', 'Health', 'Livelihoods', 'Logistics',
       'Nutrition', 'Protection', 'Shelter', 'WASH'], dtype=object)

In [9]:
sectors_risks.pillar.unique()

array(['Humanitarian Conditions', 'Impact', 'At Risk'], dtype=object)

In [10]:
sectors_risks.head()

,task,pillar,subpillar,sector,risk,risk_score,ID
40,situation_analysis_2d,Humanitarian Conditions,Living Standards,Education,Disruption of education due to schools being u...,9,"[84, 1045, 103, 2972, 2483, 1226, 2031]"
41,situation_analysis_2d,Humanitarian Conditions,Living Standards,Education,Limited access to education services for displ...,8,"[84, 2485, 1085]"
42,situation_analysis_2d,Humanitarian Conditions,Living Standards,Education,Insufficient coverage of education partners in...,7,[2485]
43,situation_analysis_2d,Humanitarian Conditions,Living Standards,Food Security,Market accessibility disruptions,7,"[3033, 3009, 3111]"
44,situation_analysis_2d,Humanitarian Conditions,Living Standards,Food Security,Reduced availability of food items,7,"[3050, 3111]"


In [11]:
SEVERITY_SCALE = {
    0:  "UNKNOWN",
    1:  "MINOR",
    2:  "MINOR",
    3:  "MODERATE",
    4:  "MODERATE",
    5:  "SERIOUS",
    6:  "SERIOUS",
    7:  "SEVERE",
    8:  "SEVERE",
    9:  "CRITICAL",
    10: "CRITICAL",
}

shown_risks = defaultdict(lambda: defaultdict(dict))
for pillar_2d in ["Impact", "Humanitarian Conditions", "At Risk"]:
    for sector in sectors_list:
        risks_one_sector_one_pillar = sectors_risks[
            (sectors_risks.pillar==pillar_2d) & 
            (sectors_risks.sector==sector)
        ]
        highest_score = risks_one_sector_one_pillar.risk_score.max()
        shown_risks[pillar_2d][sector]["highest_score"] = highest_score
        shown_risks[pillar_2d][sector]["severity_scale"] = SEVERITY_SCALE.get(highest_score, "UNKNOWN")
        top3_risks = risks_one_sector_one_pillar.sort_values(by="risk_score", ascending=False).head(3)
        shown_risks[pillar_2d][sector]["top3_risks"] = [row["risk"] for _, row in top3_risks.iterrows()]

shown_risks

defaultdict(<function __main__.<lambda>()>,
            {'Impact': defaultdict(dict,
                         {'Education': {'highest_score': 9,
                           'severity_scale': 'CRITICAL',
                           'top3_risks': ['Disruption of education due to school conversion to shelters',
                            'Damage to critical infrastructure including schools',
                            'Disruption of education services due to schools being used as collective shelters']},
                          'Food Security': {'highest_score': 8,
                           'severity_scale': 'SEVERE',
                           'top3_risks': ['Food price inflation',
                            'Supply chain disruptions',
                            'Displacement and overcrowding leading to food insecurity']},
                          'Health': {'highest_score': 9,
                           'severity_scale': 'CRITICAL',
                           'top3_risks': ['Limite

In [12]:
with open("data/WestAsia2026/analysis/Lebanon/shown_risks.json", "w") as f:
    json.dump(shown_risks, f)

# Save a JS version as well (for direct embedding in dashboard HTML)
shown_risks_js_path = "data/WestAsia2026/analysis/Lebanon/shown_risks.js"
shown_risks_json_str = json.dumps(shown_risks, ensure_ascii=False, allow_nan=True)

with open("src/viz/shown_risks_data.js", "w", encoding="utf-8") as f:
    f.write("window.SHOWN_RISKS_DATA = ")
    f.write(shown_risks_json_str)
    f.write(";")

In [13]:
key_indicator_numbers_df[key_indicator_numbers_df.pillar=="Humanitarian Access"]

,task,pillar,subpillar,sector,key_indicator,number,unit,location,specific_population,date,risk_score,ID
12,situation_analysis_1d,Humanitarian Access,People Facing Humanitarian Access Constraints-...,-,Percentage of mapped community sites covered b...,73,%,Lebanon,-,-,5,[2533]
13,situation_analysis_1d,Humanitarian Access,Physical Constraints,-,Accessibility scale rating of locations,4,scale (1-7),"Baalbek-El Hermel, Mount Lebanon, South Lebanon",-,-,6,[2197]
14,situation_analysis_1d,Humanitarian Access,Physical Constraints,-,Accessibility scale rating of locations,7,scale (1-7),"Baalbek-El Hermel, Mount Lebanon, South Lebanon",-,-,6,[2197]
15,situation_analysis_1d,Humanitarian Access,Population To Relief,-,Displaced migrants facing barriers to assistance,45000,people,Lebanon,migrants,-,6,"[224, 1686]"
16,situation_analysis_1d,Humanitarian Access,Population To Relief,-,IDPs registered with Ministry of Social Affairs,1049328,people,Lebanon,IDPs,16-03-2026,5,[1686]
17,situation_analysis_1d,Humanitarian Access,Population To Relief,-,"Departures from Lebanon (land, air, sea)",256647,people,Lebanon,-,02-03-2026 to 16-03-2026,5,"[224, 1686]"
18,situation_analysis_1d,Humanitarian Access,Relief To Population,-,Medical and ambulatory centres impacted,15,facilities,Lebanon,-,-,8,"[85, 1046]"
19,situation_analysis_1d,Humanitarian Access,Relief To Population,-,Hospitals closed,5,facilities,Lebanon,-,-,8,"[85, 1046]"
20,situation_analysis_1d,Humanitarian Access,Relief To Population,-,Attacks on emergency medical service workers,67,incidents,Lebanon,medical staff,-,9,"[85, 1046]"
21,situation_analysis_1d,Humanitarian Access,Relief To Population,-,Paramedics killed in attack,1,people,"Majdal Zoun, Lebanon",paramedics,09-03-2026,9,"[232, 1716]"


In [14]:
risks_df[risks_df.pillar=="Humanitarian Access"]

,task,pillar,subpillar,sector,risk,risk_score,ID
25,situation_analysis_1d,Humanitarian Access,People Facing Humanitarian Access Constraints-...,-,Restricted humanitarian access due to volatile...,7,"[1723, 239, 1536]"
26,situation_analysis_1d,Humanitarian Access,People Facing Humanitarian Access Constraints-...,-,Humanitarian access limitations due to adminis...,6,"[1881, 2502]"
27,situation_analysis_1d,Humanitarian Access,People Facing Humanitarian Access Constraints-...,-,Significant proportion of population not reach...,5,[2533]
28,situation_analysis_1d,Humanitarian Access,Physical Constraints,-,Inconsistent access due to security and moveme...,8,"[1068, 1536, 1175, 1723, 239]"
29,situation_analysis_1d,Humanitarian Access,Physical Constraints,-,Damage to critical infrastructure impeding hum...,7,"[1227, 967]"
30,situation_analysis_1d,Humanitarian Access,Physical Constraints,-,Poor road conditions limiting movement and ser...,6,"[967, 2197]"
31,situation_analysis_1d,Humanitarian Access,Physical Constraints,-,Volatile security conditions threatening human...,7,"[1175, 1723, 239]"
32,situation_analysis_1d,Humanitarian Access,Physical Constraints,-,Limited coordination capacity for movements to...,6,[2923]
33,situation_analysis_1d,Humanitarian Access,Population To Relief,-,Limited humanitarian access due to ongoing hos...,7,[1746]
34,situation_analysis_1d,Humanitarian Access,Population To Relief,-,Discriminatory access to shelter and assistanc...,8,[1799]


In [15]:
humanitarian_access_risks = defaultdict()

humanitarian_access_risks_df = risks_df[risks_df.pillar=="Humanitarian Access"]
humanitarian_access_subpillars = humanitarian_access_risks_df.subpillar.unique()

for subpillar in humanitarian_access_subpillars:
    one_subpillar_risks = humanitarian_access_risks_df[
        (humanitarian_access_risks_df.subpillar==subpillar) & (humanitarian_access_risks_df.risk_score>=8)
    ].risk.unique().tolist()
    one_subpillar_key_numbers_df = key_indicator_numbers_df[
        (key_indicator_numbers_df.pillar=="Humanitarian Access") & 
        (key_indicator_numbers_df.subpillar==subpillar)  & (key_indicator_numbers_df.risk_score>=8)
    ]
    one_subpillar_key_numbers = []
    for _, row in one_subpillar_key_numbers_df.iterrows():
        one_subpillar_key_numbers.append(f"**{row['key_indicator']}**: {row['risk_score']} {row['unit']}")

    if one_subpillar_risks:
        humanitarian_access_risks[subpillar] = ", ".join(one_subpillar_risks)
    if len(one_subpillar_key_numbers) > 0:
        for _, row in one_subpillar_key_numbers_df.iterrows():
            humanitarian_access_risks[row['key_indicator']] = f"{row['risk_score']} {row['unit']}"



viz_folder = "src/viz/"
os.makedirs(viz_folder, exist_ok=True)

js_content = (
    "window.HUMANITARIAN_ACCESS_DATA = "
    + json.dumps(dict(humanitarian_access_risks), ensure_ascii=False, indent=2)
    + ";\n"
)

with open(viz_folder + "humanitarian_access_data.js", "w") as f:
    f.write(js_content)

print("Saved humanitarian_access_data.js")


Saved humanitarian_access_data.js


In [16]:
numbers_2d = key_indicator_numbers_df[
    (key_indicator_numbers_df.task == "situation_analysis_2d")
    & (key_indicator_numbers_df.risk_score >= 8)
    & (key_indicator_numbers_df.number > 100)
    & ~(
        (key_indicator_numbers_df.unit == "people")
        & (key_indicator_numbers_df.number < 1_000)
    )
]

final_df = pd.DataFrame()
sectors_list = numbers_2d.sector.unique()
for sector in sectors_list:
    one_sector_df = numbers_2d[numbers_2d.sector == sector]
    units = one_sector_df.unit.unique()
    for one_unit in units:
        max_unit = one_sector_df[
            one_sector_df.unit == one_unit
        ].number.max()
        one_unit_df = one_sector_df[
            (one_sector_df.unit == one_unit)
            & (one_sector_df.number == max_unit)
        ][["sector", "key_indicator", "number", "unit"]].head(1)
        final_df = pd.concat([final_df, one_unit_df])

final_df.sort_values(by="sector")

TypeError: '>' not supported between instances of 'str' and 'int'

In [17]:
# Save key alarming numbers grouped by sector as JS for the dashboard
table = (
    final_df[["sector", "key_indicator", "number", "unit"]]
    .sort_values(by="sector")
    .drop_duplicates(subset=["sector", "key_indicator", "number", "unit"])
)

grouped = {"Sectoral Needs": []}
for _, row in table.iterrows():
    sector = row["sector"]
    grouped["Sectoral Needs"].append(
        {
            "key_indicator": row["key_indicator"],
            "number": None if pd.isna(row["number"]) else row["number"],
            "unit": row["unit"],
        }
    )

viz_folder = "src/viz/"
os.makedirs(viz_folder, exist_ok=True)

js_content = (
    "window.KEY_NUMBERS_DATA = "
    + json.dumps(grouped, ensure_ascii=False, indent=2)
    + ";\n"
)

with open(viz_folder + "key_sector_numbers_data.js", "w") as f:
    f.write(js_content)

print(f"Saved key_numbers_data.js ({len(table)} rows across {len(grouped)} sectors)")

NameError: name 'final_df' is not defined

In [18]:
shock_event = risks_df[risks_df.pillar == "Shock-Event"].reset_index(drop=True)
shock_event

,task,pillar,subpillar,sector,risk,risk_score,ID
0,situation_analysis_1d,Shock-Event,Type And Characteristics,-,Conflict-related civilian casualties and damage,8,"[2641, 2877]"
1,situation_analysis_1d,Shock-Event,Type And Characteristics,-,"Disease outbreaks (scabies, lice)",6,[1493]
2,situation_analysis_1d,Shock-Event,Type And Characteristics,-,Compound crisis including economic shock and r...,7,[503]
3,situation_analysis_1d,Shock-Event,Type And Characteristics,-,Physical damage to residential buildings,6,"[3113, 2641]"
4,situation_analysis_1d,Shock-Event,Type And Characteristics,-,Psychological distress due to conflict and dro...,7,[2877]
5,situation_analysis_1d,Shock-Event,Hazard & Threats,-,Conflict escalation and protection threats,9,"[95, 1216, 2891, 263, 2897]"
6,situation_analysis_1d,Shock-Event,Hazard & Threats,-,Internal displacement and cross-border outflows,8,"[437, 2897, 122]"
7,situation_analysis_1d,Shock-Event,Hazard & Threats,-,Health system collapse and attacks on medical ...,8,"[86, 1216, 263]"
8,situation_analysis_1d,Shock-Event,Hazard & Threats,-,Severe food insecurity and risk of acute hunger,7,"[1437, 1451]"
9,situation_analysis_1d,Shock-Event,Hazard & Threats,-,Long-term environmental and biological hazards...,8,[2891]


In [19]:
current_hazards_and_threats = risks_df[risks_df.subpillar == "Hazard & Threats"]
current_hazards_and_threats = current_hazards_and_threats.risk.unique().tolist()
current_hazards_and_threats


viz_folder = "src/viz/"
os.makedirs(viz_folder, exist_ok=True)

js_content = (
    "window.CURRENT_HAZARDS_AND_THREATS_DATA = "
    + json.dumps(current_hazards_and_threats, ensure_ascii=False, indent=2)
    + ";\n"
)

with open(viz_folder + "current_hazards_and_threats_data.js", "w") as f:
    f.write(js_content)

print(f"Saved current_hazards_and_threats_data.js ({len(current_hazards_and_threats)} items)")


Saved current_hazards_and_threats_data.js (5 items)


In [20]:
precrisis_vulnerabilities = risks_df[risks_df.subpillar == "Underlying-Aggravating Factors"]
precrisis_vulnerabilities = precrisis_vulnerabilities.risk.unique().tolist()
precrisis_vulnerabilities

['Overstretched fragile services',
 'Deteriorating security conditions',
 'Overwhelmed emergency healthcare capacity',
 'Rapid displacement crisis',
 'Escalation to total war']

In [21]:
viz_folder = "src/viz/"
os.makedirs(viz_folder, exist_ok=True)

js_content = (
    "window.PRECRISIS_VULNERABILITIES_DATA = "
    + json.dumps(precrisis_vulnerabilities, ensure_ascii=False, indent=2)
    + ";\n"
)

with open(viz_folder + "precrisis_vulnerabilities_data.js", "w") as f:
    f.write(js_content)

print(f"Saved precrisis_vulnerabilities_data.js ({len(precrisis_vulnerabilities)} items)")


Saved precrisis_vulnerabilities_data.js (5 items)


In [22]:
displacement_risks_df = risks_df[risks_df.pillar == "Displacement"]
displacemnt_data = defaultdict(list)
for subpillar in ["Push Factors", "Intentions"]:
    one_subpillar_risks = displacement_risks_df[
        (displacement_risks_df.subpillar == subpillar)
    ].sort_values(by="risk_score", ascending=False).head(4).risk.unique().tolist()
    displacemnt_data[subpillar] = one_subpillar_risks

displacemnt_data

viz_folder = "src/viz/"
os.makedirs(viz_folder, exist_ok=True)

js_content = (
    "window.DISPLACEMENT_DATA = "
    + json.dumps(displacemnt_data, ensure_ascii=False, indent=2)
    + ";\n"
)

with open(viz_folder + "displacement_data.js", "w") as f:
    f.write(js_content)

print(f"Saved displacement_data.js ({len(displacemnt_data)} items)")


Saved displacement_data.js (2 items)


In [23]:
key_indicator_numbers_df[key_indicator_numbers_df.pillar == "Displacement"]

,task,pillar,subpillar,sector,key_indicator,number,unit,location,specific_population,date,risk_score,ID
0,situation_analysis_1d,Displacement,Push Factors,-,Territory under evacuation orders,14,%,Lebanon,-,-,8,[2436]
1,situation_analysis_1d,Displacement,Push Factors,-,Territory under evacuation orders,20,%,Lebanon,-,-,8,[1496]
2,situation_analysis_1d,Displacement,Intentions,-,Persons crossing from Lebanon into Syria,147800,people,Lebanon-Syria border,displaced Syrians and others,-,6,[1659]
3,situation_analysis_1d,Displacement,Intentions,-,Syrians indicating intention to return permane...,39840,people,Lebanon-Syria border,Syrian refugees,-,6,[1659]
4,situation_analysis_1d,Displacement,Local Integration,-,Rising community tensions,-,-,Lebanon,displaced populations and host communities,-,7,"[2181, 2399]"


In [39]:
with open("data/WestAsia2026/analysis/Lebanon/priority_needs.json", "r") as f:
    priority_needs = json.load(f)

priority_needs = pd.DataFrame(priority_needs)

top_sectoral_needs = priority_needs[
    (priority_needs.task == "situation_analysis_2d") & 
    (priority_needs.priority_need_score >= 9)
].sort_values(by="priority_need_score", ascending=False)

top_sectoral_needs_dict = defaultdict(list)
for _, row in top_sectoral_needs.iterrows():
    top_sectoral_needs_dict[row["sector"]].append(row["priority_need"])

top_sectoral_needs_dict

viz_folder = "src/viz/"
os.makedirs(viz_folder, exist_ok=True)

js_content = (
    "window.TOP_SECTORAL_NEEDS_DATA = "
    + json.dumps(top_sectoral_needs_dict, ensure_ascii=False, indent=2)
    + ";\n"
)

with open(viz_folder + "top_sectoral_needs_data.js", "w") as f:
    f.write(js_content)

print(f"Saved top_sectoral_needs_data.js ({len(top_sectoral_needs_dict)} items)")


Saved top_sectoral_needs_data.js (8 items)


In [45]:
with open("data/WestAsia2026/analysis/Lebanon/priority_interventions.json", "r") as f:
    priority_interventions = json.load(f)

priority_interventions = pd.DataFrame(priority_interventions)

top_priority_interventions = priority_interventions[
    # (priority_interventions.task == "situation_analysis_2d") & 
    (priority_interventions.priority_intervention_score >= 9)
].sort_values(by="priority_intervention_score", ascending=False)

top_priority_interventions_dict = defaultdict(list)
for _, row in top_priority_interventions.iterrows():
    if row["sector"] == "-":
        field = row["pillar"]
    else:
        field = row["sector"]
    top_priority_interventions_dict[field].append(row["priority_intervention"])

top_priority_interventions_dict

viz_folder = "src/viz/"
os.makedirs(viz_folder, exist_ok=True)

js_content = (
    "window.TOP_PRIORITY_INTERVENTIONS_DATA = "
    + json.dumps(top_priority_interventions_dict, ensure_ascii=False, indent=2)
    + ";\n"
)

with open(viz_folder + "top_priority_interventions_data.js", "w") as f:
    f.write(js_content)

print(f"Saved top_priority_interventions_data.js ({len(top_priority_interventions_dict)} items)")

Saved top_priority_interventions_data.js (11 items)
